# Global Description Generator Test

This notebook tests only `GlobalDescriptionGenerator` against the existing `test_db5.db` graph. The database contains a completed source graph whose `source.description` is currently empty.

In [1]:
import asyncio
import json

import aiosqlite
from dotenv import load_dotenv

from dotvectordb.graph.generate_global_desc import GlobalDescriptionGenerator
from dotvectordb.plugins.llm_providers.openai_llms import OpenAiAgent

load_dotenv()
DATABASE_PATH = "test_db4.db"
SAMPLE_SIZE = 10

In [2]:
conn = await aiosqlite.connect(DATABASE_PATH)
lock = asyncio.Lock()
language_model = OpenAiAgent()

generator = GlobalDescriptionGenerator(
    conn=conn,
    lock=lock,
    language_model=language_model.generate,
    verbose=True,
)

In [3]:
cursor = await conn.execute(
    "SELECT id, name, description, num_chunks FROM source"
)
sources_before = await cursor.fetchall()
await cursor.close()

print("Sources before generation:")
for source in sources_before:
    print(source)

Sources before generation:
('-8917888661152971970', 'tests/shrek.txt', '{"global_description": "The full corpus appears to center around the whimsical and transformative themes found in the narratives associated with the Shrek franchise. It explores the interplay of love, personal growth, and significant life experiences, often set against a backdrop of fictional locations inspired by fairy tales. The characters, ranging from traditional fairy tale figures to original creations, navigate quests for love and identity, often with humor and subversion of typical tropes. The narrative emphasizes the importance of inner beauty and acceptance, while also highlighting the dynamics of friendship and camaraderie through various organizations and groups. Additionally, the corpus incorporates unique and imaginative culinary elements, reflecting the cultural significance of food within these fantastical stories. Overall, it presents a rich tapestry of emotional and philosophical themes, inviting a

In [4]:
results = await generator.generate(sample_size=SAMPLE_SIZE)
results

Found 1 source(s) for global description generation.
Source -8917888661152971970 has 6 entity type(s): concept, creative_work, location, organization, person, substance
Sending 6 entity-type description request(s) concurrently.
Sampled 3 concept entity/entities from source -8917888661152971970.
Prompt sent to the LLM for entity type 'concept':

<Role>
You are a Knowledge Graph Specialist responsible for reading a list of randomly sampled entities of a specific type with the objective
of trying to figure out a global description of what the full corpus could be about.
</Role>

<Instructions>
   - Use the source name in the '<Source_Name>' section as context for identifying what the source document is about.
   - Carefully read the '<Type>' section to understand the type of entities that were sampled from the graph database.
   - Read the '<Sample>' section to understand the sampled entities of the specific type.
   - Based on the information in the '<Type>' and '<Sample>' sections, obse

{'-8917888661152971970': {'global_description': 'The corpus centers around the animated film "Shrek" and its exploration of themes commonly found in fairy tales and romantic narratives. It delves into significant concepts such as love, transformation, and personal growth, often framed within a whimsical and fantastical narrative. The characters, including Shrek, Princess Fiona, and Donkey, challenge traditional fairy tale tropes, emphasizing friendship, identity, and the importance of inner beauty over societal expectations. The narrative is enriched by various fictional locations, such as Shrek\'s swamp and a castle, which serve as backdrops for pivotal events and character journeys. Additionally, the corpus features humorous and unconventional culinary elements that reflect the quirky personalities of the characters. Overall, it presents a comedic yet profound examination of love, adventure, and the quest for acceptance in a magical world.',
  'entity_type_descriptions': {'concept': 

In [5]:
cursor = await conn.execute(
    "SELECT id, name, description FROM source"
)
sources_after = await cursor.fetchall()
await cursor.close()

for source_id, source_name, description in sources_after:
    print(f"Source: {source_name} ({source_id})")
    print(json.dumps(json.loads(description), indent=2))

Source: tests/shrek.txt (-8917888661152971970)
{
  "global_description": "The full corpus appears to center around the themes of love, happiness, and transformative experiences, often depicted in fairy tales and romantic narratives. It explores significant moments that define relationships, such as 'true love's first kiss' and the pursuit of 'happily ever after,' alongside challenges that characters must overcome to achieve fulfillment. The corpus encompasses a variety of creative works, including music, literature, and visual arts, that resonate with audiences through relatable emotional experiences. Additionally, it features fictional locations and organizations, particularly from the realm of fantasy and folklore, highlighting the dynamics of whimsical worlds and the adventures of their inhabitants. Characters from these narratives, such as those in the \"Shrek\" franchise, embody themes of friendship, adventure, and the struggle between good and evil. The exploration of unique and 

In [ ]:
await conn.close()
print("Database connection closed.")